# 2 Build Match Rate Flags
Compute enhanced combined match condition across Scrape, Yelp, and Yellow Pages signals.
Isolate OCR rows that fail all match criteria and write them to temp for review.

Input:  data/1_derived/4_scrape_yellowpages/1_yellowpages_phone_matched.csv
Output: data/1_derived/4_scrape_yellowpages/2_match_rates.csv
        temp/merge_yellowpages/isolated_rows.csv

In [ ]:
from pathlib import Path
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'source').exists() and (candidate / 'temp').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
YP_DIR = PROJECT_ROOT / 'data' / '1_derived' / '4_scrape_yellowpages'
IN_FILE = YP_DIR / '1_yellowpages_phone_matched.csv'
OUT_FILE = YP_DIR / '2_match_rates.csv'
ISOLATED_FILE = PROJECT_ROOT / 'temp' / 'merge_yellowpages' / 'isolated_rows.csv'
ISOLATED_FILE.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(IN_FILE, low_memory=False)
print(f'Loaded: {len(df):,} rows')

# Enhanced combined condition: row is considered "matched" if ANY of these succeed
enhanced_combined_condition = (
    (df['Scraped_phone_match_rate'] == True) |
    (df['Scraped_zipcode_to_label_match_rate'].isin(['6/6 successful match', '7/6 successful match'])) |
    (df['Yelp_phone_match_rate'] == True) |
    (df['Yellowbook_phone_match_rate'] == True)
)

df['enhanced_match'] = enhanced_combined_condition

matched = enhanced_combined_condition.sum()
unmatched = (~enhanced_combined_condition).sum()
print(f'\nRows meeting enhanced criteria  : {matched:,} ({matched/len(df)*100:.2f}%)')
print(f'Rows NOT meeting any criteria   : {unmatched:,} ({unmatched/len(df)*100:.2f}%)')

# Save full dataset with the flag
df.to_csv(OUT_FILE, index=False)
print(f'\nSaved match rates: {OUT_FILE}')

# Save the unmatched rows separately for review
isolated = df[~enhanced_combined_condition].copy()
isolated.to_csv(ISOLATED_FILE, index=False)
print(f'Saved isolated rows: {ISOLATED_FILE}  ({len(isolated):,} rows)')

df[['label', 'place_identifier(year)', 'enhanced_match']].head(10)

Saved: c:\Users\Owner\Desktop\Geocoding_Truck_Stops\data\1_derived\scrape_yelp_truckstops\2_match_rates.csv
Yelp_phone_match_rate
True     10687
False     5002
Name: count, dtype: int64
